# Pokaa Scraper — Sorties par date d'événement
Entrez une date dans la cellule suivante et exécutez toutes les cellules.

In [ ]:
# ✏️ MODIFIE LA DATE ICI
DATE_CIBLE = "07/03/2026"  # formats acceptés : DD/MM/YYYY ou YYYY-MM-DD

: 

In [ ]:
import sys
import re
import requests
from bs4 import BeautifulSoup
from datetime import datetime

# ── Configuration ───────────────────────────────────────────────────────────────────
DATE_FORMATS = ["%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y", "%d.%m.%Y"]
BASE_URL = "https://pokaa.fr/categories/ses-sorties/"
MAX_PAGES = 5

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "fr-FR,fr;q=0.9",
    "Referer": "https://www.google.com/",
}

MONTHS_FR = {
    "janvier": 1, "f\u00e9vrier": 2, "fevrier": 2, "mars": 3, "avril": 4,
    "mai": 5, "juin": 6, "juillet": 7, "ao\u00fbt": 8, "aout": 8,
    "septembre": 9, "octobre": 10, "novembre": 11, "d\u00e9cembre": 12, "decembre": 12
}

print("Configuration chargée")

In [ ]:
# ── Fonctions ────────────────────────────────────────────────────────────────

def parse_input_date(raw):
    for fmt in DATE_FORMATS:
        try:
            return datetime.strptime(raw.strip(), fmt)
        except ValueError:
            continue
    raise ValueError(f"Format non reconnu : '{raw}'  →  Essaie : 07/03/2026 ou 2026-03-07")

def extract_event_dates(text, ref_year):
    text_lower = text.lower()
    results = []

    # "du JJ [mois] au JJ mois"
    p_range = re.finditer(
        r'du\s+(\d{1,2})\s*(?:er)?\s*(janvier|f\u00e9vrier|fevrier|mars|avril|mai|juin|juillet|ao\u00fbt|aout|septembre|octobre|novembre|d\u00e9cembre|decembre)?\s+'
        r'(?:\d+h\d*\s+)?au\s+(\d{1,2})\s*(?:er)?\s*(janvier|f\u00e9vrier|fevrier|mars|avril|mai|juin|juillet|ao\u00fbt|aout|septembre|octobre|novembre|d\u00e9cembre|decembre)',
        text_lower
    )
    for m in p_range:
        d1, mo1, d2, mo2 = m.group(1), m.group(2), m.group(3), m.group(4)
        month2 = MONTHS_FR.get(mo2)
        month1 = MONTHS_FR.get(mo1) if mo1 else month2
        if month1 and month2:
            try:
                results.append((datetime(ref_year, month1, int(d1)), datetime(ref_year, month2, int(d2))))
            except ValueError:
                pass

    # "JJ et JJ mois"
    p_and = re.finditer(
        r'(\d{1,2})\s+et\s+(\d{1,2})\s+(janvier|f\u00e9vrier|fevrier|mars|avril|mai|juin|juillet|ao\u00fbt|aout|septembre|octobre|novembre|d\u00e9cembre|decembre)',
        text_lower
    )
    for m in p_and:
        d1, d2, mo = m.group(1), m.group(2), m.group(3)
        month = MONTHS_FR.get(mo)
        if month:
            try:
                results.append((datetime(ref_year, month, int(d1)), datetime(ref_year, month, int(d1))))
                results.append((datetime(ref_year, month, int(d2)), datetime(ref_year, month, int(d2))))
            except ValueError:
                pass

    # "vendredi JJ mois" / "le JJ mois"
    p_single = re.finditer(
        r'(?:lundi|mardi|mercredi|jeudi|vendredi|samedi|dimanche|le)?\s*(\d{1,2})\s*(?:er)?\s+'
        r'(janvier|f\u00e9vrier|fevrier|mars|avril|mai|juin|juillet|ao\u00fbt|aout|septembre|octobre|novembre|d\u00e9cembre|decembre)',
        text_lower
    )
    for m in p_single:
        d, mo = m.group(1), m.group(2)
        month = MONTHS_FR.get(mo)
        if month:
            try:
                dt = datetime(ref_year, month, int(d))
                already = any(r[0] <= dt <= r[1] for r in results if r[1])
                if not already:
                    results.append((dt, dt))
            except ValueError:
                pass
    return results

def date_is_covered(target, event_dates):
    for (d1, d2) in event_dates:
        if d2 is None:
            if d1.date() == target.date():
                return True
        else:
            if d1.date() <= target.date() <= d2.date():
                return True
    return False

def fetch_soup(session, url):
    resp = session.get(url, headers=HEADERS, timeout=15)
    if resp.status_code == 404:
        return None
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")

def get_listing_articles(soup):
    articles = []
    for art in soup.find_all("article"):
        link_tag = art.find("a", href=True)
        lien = link_tag["href"] if link_tag else ""
        title_tag = art.find(["h2", "h3"])
        titre = title_tag.get_text(strip=True) if title_tag else "Sans titre"
        cat_tag = art.find(class_="post-category")
        categorie = cat_tag.get_text(strip=True) if cat_tag else ""
        pub_date = None
        m = re.search(r"/(\d{4})/(\d{2})/(\d{2})/", lien)
        if m:
            try:
                pub_date = datetime(int(m.group(1)), int(m.group(2)), int(m.group(3)))
            except ValueError:
                pass
        articles.append({"titre": titre, "lien": lien, "categorie": categorie, "pub_date": pub_date})
    return articles

def get_article_event_dates(session, url, ref_year):
    try:
        soup = fetch_soup(session, url)
        if not soup:
            return []
        content = soup.find("div", class_="entry-content")
        if not content:
            return []
        text = content.get_text(" ", strip=True)
        return extract_event_dates(text, ref_year)
    except Exception as e:
        print(f"      ⚠️  Erreur : {e}")
        return []

print("✅ Fonctions chargées")

In [ ]:
# ── Lancement du scraper ─────────────────────────────────────────────────────

target = parse_input_date(DATE_CIBLE)
ref_year = target.year
print(f"\n🔍 Recherche des événements pour le {target.strftime('%d/%m/%Y')}...\n")

session = requests.Session()
matched = []

for page in range(1, MAX_PAGES + 1):
    url = BASE_URL if page == 1 else f"{BASE_URL}page/{page}/"
    print(f"→ Page {page}...")
    soup = fetch_soup(session, url)
    if not soup:
        print("  Fin de pagination.")
        break

    articles = get_listing_articles(soup)
    if not articles:
        break

    seen_liens = set(r["lien"] for r in matched)

    for art in articles:
        if art["lien"] in seen_liens:
            continue
        if art["categorie"].lower() not in ("sorties", "sortie"):
            continue

        print(f"   🔎 {art['titre'][:70]}...")
        event_dates = get_article_event_dates(session, art["lien"], ref_year)
        unique_dates = list(dict.fromkeys(event_dates))

        if unique_dates:
            dates_str = ", ".join(
                f"{d1.strftime('%d/%m')}→{d2.strftime('%d/%m')}" if d1 != d2 else d1.strftime('%d/%m')
                for d1, d2 in unique_dates
            )
        else:
            dates_str = "dates non détectées"

        if date_is_covered(target, unique_dates):
            matched.append({**art, "dates_str": dates_str})
            seen_liens.add(art["lien"])

# ── Affichage des résultats ──────────────────────────────────────────────────
date_str = target.strftime("%d/%m/%Y")
print(f"\n{'='*60}")
print(f"  🎉 Sorties Pokaa à découvrir — {date_str}")
print(f"{'='*60}")

if not matched:
    print(f"\n  Aucun événement trouvé pour le {date_str}.")
    print("  → Pokaa n'a peut-être pas publié d'agenda couvrant cette date.")
else:
    for i, r in enumerate(matched, 1):
        print(f"\n[{i}] {r['titre']}")
        if r.get("categorie"):
            print(f"    🏷️  {r['categorie']}")
        print(f"    📅 Dates : {r['dates_str']}")
        print(f"    🔗 {r['lien']}")
    print(f"\n{'='*60}")
    print(f"  ✅ {len(matched)} article(s) trouvé(s) pour le {date_str}")
    print(f"{'='*60}")